In [1]:
import cv2
import mediapipe as mp
import pandas as pd
import os
import numpy as np

In [2]:
# Configuración de carpetas
video_folder = "videos_recortados/user_id_24026"
output_folder = "landmarks_csvs_recortados"
os.makedirs(output_folder, exist_ok=True)

In [3]:
# Inicializar el modelo de MediaPipe
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1)

In [ ]:
# Procesamiento de vídeos
for filename in os.listdir(video_folder):
    if filename.endswith(".mp4"):
        video_path = os.path.join(video_folder, filename)
        output_csv = os.path.join(output_folder, filename.replace(".mp4", ".csv"))
        
        print(f"Probando abrir {filename}...")
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"ERROR: No se pudo abrir {filename}")
            continue  # Salta al siguiente vídeo

        print(f"Procesando {filename}...")

        frame_number = 0
        all_landmarks = []

        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_frame)

            if results.multi_face_landmarks:
                for face_landmarks in results.multi_face_landmarks:
                    frame_data = {"frame": frame_number}
                    for i, lm in enumerate(face_landmarks.landmark):
                        frame_data[f"x_{i}"] = lm.x
                        frame_data[f"y_{i}"] = lm.y
                        frame_data[f"z_{i}"] = lm.z
                    all_landmarks.append(frame_data)

            frame_number += 1

            # Si se detectan landmarks, los dibuja
            if results.multi_face_landmarks:
                for face_landmarks in results.multi_face_landmarks:
                    mp_drawing.draw_landmarks(
                        image=frame,
                        landmark_list=face_landmarks,
                        connections=mp_face_mesh.FACEMESH_TESSELATION,
                        landmark_drawing_spec=None,
                        connection_drawing_spec=mp_drawing_styles
                        .get_default_face_mesh_tesselation_style()
                    )

            cv2.imshow('FaceMesh', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        
        if all_landmarks:  # Solo guarda si hay landmarks
            df = pd.DataFrame(all_landmarks)
            df.to_csv(output_csv, index=False)
            print(f"Guardado: {output_csv}")
        else:
            print(f"No se encontraron landmarks en {filename}")

[→] Probando abrir Dashboard_User_id_24026_3_recortado.mp4...
[✓] Procesando Dashboard_User_id_24026_3_recortado.mp4...


c:\anaconda3\envs\gpu\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


[✓] Guardado: landmarks_csvs_recortados\Dashboard_User_id_24026_3_recortado.csv
[→] Probando abrir Dashboard_User_id_24026_4_recortado.mp4...
[✓] Procesando Dashboard_User_id_24026_4_recortado.mp4...
[✓] Guardado: landmarks_csvs_recortados\Dashboard_User_id_24026_4_recortado.csv
[→] Probando abrir Rear_view_User_id_24026_3_recortado.mp4...
[✓] Procesando Rear_view_User_id_24026_3_recortado.mp4...
[✓] Guardado: landmarks_csvs_recortados\Rear_view_User_id_24026_3_recortado.csv
[→] Probando abrir Rear_view_User_id_24026_4_recortado.mp4...
[✓] Procesando Rear_view_User_id_24026_4_recortado.mp4...
[✓] Guardado: landmarks_csvs_recortados\Rear_view_User_id_24026_4_recortado.csv
[→] Probando abrir Right_side_window_User_id_24026_3_recortado.mp4...
[✓] Procesando Right_side_window_User_id_24026_3_recortado.mp4...
[!] No se encontraron landmarks en Right_side_window_User_id_24026_3_recortado.mp4
[→] Probando abrir Right_side_window_User_id_24026_4_recortado.mp4...
[✓] Procesando Right_side_windo

In [5]:
face_mesh.close()